# Set Covering Problem — Constructive Heuristics
### Exercise 1.1 — Data reading & instance representation

This notebook is the foundation for the assignment *"Constructive heuristics for the SCP"*.

In this first part we build:
1. A parser for the OR-Library SCP instance format.
2. An `SCPInstance` class that stores the problem and exposes the incremental
   bookkeeping (covered elements, coverage counts, candidate columns, cost)
   that CH1–CH3 and the redundancy-elimination procedure will need later.
3. A small sanity check loading the two example instances (`scp57.txt`,
   `scp58.txt`) and comparing basic stats against the best-known solutions
   table given in the assignment.

Later notebooks/sections will add: CH1, CH2, CH3, redundancy elimination (RE),
and the experimental comparison (average % deviation from BKS, fraction of
instances improved by RE).

## 1. The OR-Library SCP file format

Each instance file is a whitespace-separated stream of integers, laid out as:

```
m n
c_1 c_2 ... c_n              # cost of every column (set), n values
k_1 <k_1 column indices>     # row 1: how many columns cover it, then which ones (1-based)
k_2 <k_2 column indices>     # row 2
...
k_m <k_m column indices>     # row m
```

- `m` = number of elements (rows) that must be covered
- `n` = number of available sets (columns)
- Column indices are **1-based** in the file.

We verified this on `scp57.txt`: header `200 2000`, followed by exactly 2000
cost values (all between 1 and 100), followed by 200 rows whose per-row
column lists consume the file exactly to the last token.

In [1]:
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Set, Dict
import random

DATA_DIR = Path("data")

## 2. Parser

In [2]:
def read_scp_instance(filepath):
    """Parse an OR-Library formatted SCP instance file.

    Returns a dict with:
      m        : int, number of elements (rows)
      n        : int, number of sets (columns)
      costs    : np.ndarray shape (n,), cost of each set (0-indexed sets)
      rows     : list[list[int]] length m, rows[i] = 0-indexed sets covering element i
      cols     : list[list[int]] length n, cols[j] = 0-indexed elements covered by set j
                 (built by inverting `rows`, convenient for incremental updates)
    """
    with open(filepath, "r") as f:
        tokens = f.read().split()
    tokens = list(map(int, tokens))

    idx = 0
    m, n = tokens[idx], tokens[idx + 1]
    idx += 2

    costs = np.array(tokens[idx: idx + n], dtype=float)
    idx += n

    rows = []
    for _ in range(m):
        k = tokens[idx]; idx += 1
        # convert 1-based file indices to 0-based
        cols_for_row = [c - 1 for c in tokens[idx: idx + k]]
        idx += k
        rows.append(cols_for_row)

    assert idx == len(tokens), f"Parser did not consume the whole file ({idx} vs {len(tokens)} tokens)"

    # invert rows -> cols (which elements each set covers)
    cols = [[] for _ in range(n)]
    for row_idx, covering_sets in enumerate(rows):
        for s in covering_sets:
            cols[s].append(row_idx)

    return {"m": m, "n": n, "costs": costs, "rows": rows, "cols": cols}

## 3. `SCPInstance` — problem wrapper with incremental state

This class holds the static problem data plus the *mutable* state of a
partial solution under construction:

- `covered`: boolean mask of covered elements
- `cover_count`: how many chosen sets currently cover each element (needed to
  detect, during redundancy elimination, whether removing a set leaves some
  element uncovered)
- `chosen`: boolean mask of sets already in the solution
- `remaining_gain(j)`: number of *currently uncovered* elements that set `j`
  would newly cover — this is exactly the quantity CH1–CH3 need to (a) decide
  whether a set is redundant (`remaining_gain == 0` ⇒ skip it) and (b) drive
  greedy choices in later heuristics, all updated incrementally instead of
  recomputed from scratch at every step.

In [3]:
class SCPInstance:
    def __init__(self, data, name=None):
        self.name = name
        self.m = data["m"]
        self.n = data["n"]
        self.costs = data["costs"]
        self.rows = data["rows"]   # element -> covering sets
        self.cols = data["cols"]   # set -> covered elements
        self.reset()

    def reset(self):
        """Reset to an empty partial solution."""
        self.covered = np.zeros(self.m, dtype=bool)
        self.cover_count = np.zeros(self.m, dtype=int)
        self.chosen = np.zeros(self.n, dtype=bool)
        self.num_covered = 0
        self.solution_cost = 0.0

    def uncovered_elements(self):
        return np.flatnonzero(~self.covered)

    def remaining_gain(self, j):
        """# of currently uncovered elements set j would newly cover."""
        if self.chosen[j]:
            return 0
        return sum(1 for e in self.cols[j] if not self.covered[e])

    def is_redundant(self, j):
        """True if adding set j would cover no new element."""
        return self.remaining_gain(j) == 0

    def add_set(self, j):
        """Add set j to the solution and incrementally update all state."""
        if self.chosen[j]:
            return 0
        self.chosen[j] = True
        self.solution_cost += self.costs[j]
        newly_covered = 0
        for e in self.cols[j]:
            self.cover_count[e] += 1
            if not self.covered[e]:
                self.covered[e] = True
                self.num_covered += 1
                newly_covered += 1
        return newly_covered

    def remove_set(self, j):
        """Remove set j (used by redundancy elimination). Only valid if,
        after removal, every element is still covered by >=1 other set."""
        if not self.chosen[j]:
            return
        self.chosen[j] = False
        self.solution_cost -= self.costs[j]
        for e in self.cols[j]:
            self.cover_count[e] -= 1
            if self.cover_count[e] == 0:
                self.covered[e] = False
                self.num_covered -= 1

    def is_complete_cover(self):
        return self.num_covered == self.m

    def solution_sets(self):
        return np.flatnonzero(self.chosen).tolist()

    def __repr__(self):
        return f"SCPInstance(name={self.name}, m={self.m}, n={self.n})"

## 4. Load the example instances

In [4]:
instances = {}
for fname, label in [("scp57.txt", "5.7"), ("scp58.txt", "5.8")]:
    data = read_scp_instance(DATA_DIR / fname)
    inst = SCPInstance(data, name=label)
    instances[label] = inst
    print(f"{label}: m={inst.m} elements, n={inst.n} sets, "
          f"cost range=[{inst.costs.min():.0f}, {inst.costs.max():.0f}], "
          f"avg sets/element={np.mean([len(r) for r in inst.rows]):.1f}")

5.7: m=200 elements, n=2000 sets, cost range=[1, 100], avg sets/element=40.3
5.8: m=200 elements, n=2000 sets, cost range=[1, 100], avg sets/element=39.6


## 5. Best-known solutions (for later % deviation reporting)

Taken from the values given in the assignment. Keyed by instance label
(matching the naming used for the OR-Library SCP instances, e.g. `scp41`
through `scp410`, `scp51`...`scp59`, `scp61`...`scp65`, `scpa1`...`scpd5`).

In [5]:
BKS = {
    "4.1": None,  # not given in the assignment text (starts at 4.2)
    "4.2": 512, "4.3": 516, "4.4": 494, "4.5": 512,
    "4.6": 560, "4.7": 430, "4.8": 492, "4.9": 641,
    "5.1": 253, "5.2": 302, "5.3": 226, "5.4": 242, "5.5": 211,
    "5.6": 213, "5.7": 293, "5.8": 288, "5.9": 279,
    "6.1": 138, "6.2": 146, "6.3": 145, "6.4": 131, "6.5": 161,
    "A.1": 253, "A.2": 252, "A.3": 232, "A.4": 234, "A.5": 236,
    "B.1": 69,  "B.2": 76,  "B.3": 80,  "B.4": 79,  "B.5": 72,
    "C.1": 227, "C.2": 219, "C.3": 243, "C.4": 219, "C.5": 215,
    "D.1": 60,  "D.2": 66,  "D.3": 72,  "D.4": 62,  "D.5": 61,
}
BKS = {k: v for k, v in BKS.items() if v is not None}

for label, inst in instances.items():
    print(f"Instance {label}: best known = {BKS.get(label, 'N/A')}")

Instance 5.7: best known = 293
Instance 5.8: best known = 288


## 6. Sanity check

Quick smoke test: greedily add sets in index order until the cover is
complete, using only the incremental API above (no heuristic quality claimed
here — this just exercises `add_set`, `remaining_gain`, `is_complete_cover`
before CH1–CH3 are implemented).

In [6]:
def naive_sequential_cover(inst):
    inst.reset()
    for j in range(inst.n):
        if inst.is_complete_cover():
            break
        if not inst.is_redundant(j):
            inst.add_set(j)
    assert inst.is_complete_cover(), "Failed to cover all elements!"
    return inst.solution_cost, len(inst.solution_sets())

for label, inst in instances.items():
    cost, n_sets = naive_sequential_cover(inst)
    bks = BKS.get(label)
    dev = (cost - bks) / bks * 100 if bks else float("nan")
    print(f"{label}: naive cost={cost:.0f}  ({n_sets} sets)  "
          f"BKS={bks}  dev={dev:.1f}%")

5.7: naive cost=423  (99 sets)  BKS=293  dev=44.4%
5.8: naive cost=422  (101 sets)  BKS=288  dev=46.5%


---
**Next steps** (not yet implemented in this notebook):

- `CH1`: pick an uncovered element at random, then a random covering set.
- `CH2`, `CH3`: two further constructive rules (e.g. greedy best-cost-per-new-element, and a second variant of your choice).
- `RE`: redundancy elimination on a complete cover (remove sets that have become unnecessary once the cover is complete).
- Run CH1–CH3 with a fixed random seed, with and without RE, on every instance, and report the average % deviation from BKS and the fraction of instances that improved thanks to RE.